In [1]:
import sys
sys.path.append('../') 
from visualisation import *
import xarray as xr
import dask
import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import KDTree
# crs = EPSG:4326 (WGS 84)

In [39]:
bom_path = "/home/hossein/CICCADA/BOM_NCI/2023/01/01/"
files = glob(bom_path+"*.nc")
len(files)

103

In [52]:
df = [xr.open_dataset(file).to_dataframe() for file in files[:15]]
df = pd.concat(df, axis=0).reset_index(drop=False)
df = df.dropna(subset='direct_normal_irradiance').reset_index(drop=True)
df['julian_date'] = pd.to_datetime(df['julian_date'], origin='julian', unit='D')
df = df[['latitude', 'longitude']].drop_duplicates().reset_index(drop=True)

In [ ]:
os.getcwd()

In [7]:
naf_path = "G-NAF/G-NAF MAY 2025/Standard/"
naf_files = glob(f"{naf_path}SA*.psv")
naf_files

['G-NAF/G-NAF MAY 2025/Standard/SA_ADDRESS_MESH_BLOCK_2016_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_STREET_LOCALITY_ALIAS_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_MB_2016_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_ADDRESS_MESH_BLOCK_2021_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_ADDRESS_DEFAULT_GEOCODE_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_LOCALITY_NEIGHBOUR_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_STATE_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_ADDRESS_FEATURE_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_PRIMARY_SECONDARY_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_MB_2021_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_ADDRESS_SITE_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_STREET_LOCALITY_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_ADDRESS_ALIAS_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_STREET_LOCALITY_POINT_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_ADDRESS_DETAIL_psv.psv',
 'G-NAF/G-NAF MAY 2025/Standard/SA_LOCALITY_POINT_psv.psv',
 'G-NAF/G-NAF MA

In [ ]:
SA_STREET_LOCALITY_POINT_psv = pd.read_csv(glob(f"{naf_path}SA_STREET_LOCALITY_POINT_psv.psv")[0], sep='|', low_memory=False).dropna(axis=1)
SA_ADDRESS_DETAIL_psv = pd.read_csv(glob(f"{naf_path}SA_ADDRESS_DETAIL_psv.psv")[0], sep='|', low_memory=False).dropna(axis=1)
# SA_ADDRESS_DETAIL_psv


In [5]:
SA_STREET_LOCALITY_POINT_psv[:2]

,STREET_LOCALITY_POINT_PID,DATE_CREATED,STREET_LOCALITY_PID,LONGITUDE,LATITUDE
0,L7125736,2018-08-04,SA511308,138.817210,-34.890257
1,L7047908,2020-08-06,SA506498,138.828976,-34.992895


In [6]:
SA_ADDRESS_DETAIL_psv[:2]

,ADDRESS_DETAIL_PID,DATE_CREATED,DATE_LAST_MODIFIED,STREET_LOCALITY_PID,LOCALITY_PID,ALIAS_PRINCIPAL,POSTCODE,CONFIDENCE,ADDRESS_SITE_PID,LEVEL_GEOCODED_CODE
0,GASA_424134104,2008-10-01,2021-07-07,SA528832,loc747214e0548c,P,5275,1,424221699,7
1,GASA_424134105,2008-10-01,2021-07-07,SA528832,loc747214e0548c,P,5275,1,424221700,7


In [11]:
a = pd.read_csv(glob(f"{naf_path}SA_ADDRESS_DETAIL_psv.psv")[0], sep='|', low_memory=False)

In [12]:
a.columns

Index(['ADDRESS_DETAIL_PID', 'DATE_CREATED', 'DATE_LAST_MODIFIED',
       'DATE_RETIRED', 'BUILDING_NAME', 'LOT_NUMBER_PREFIX', 'LOT_NUMBER',
       'LOT_NUMBER_SUFFIX', 'FLAT_TYPE_CODE', 'FLAT_NUMBER_PREFIX',
       'FLAT_NUMBER', 'FLAT_NUMBER_SUFFIX', 'LEVEL_TYPE_CODE',
       'LEVEL_NUMBER_PREFIX', 'LEVEL_NUMBER', 'LEVEL_NUMBER_SUFFIX',
       'NUMBER_FIRST_PREFIX', 'NUMBER_FIRST', 'NUMBER_FIRST_SUFFIX',
       'NUMBER_LAST_PREFIX', 'NUMBER_LAST', 'NUMBER_LAST_SUFFIX',
       'STREET_LOCALITY_PID', 'LOCATION_DESCRIPTION', 'LOCALITY_PID',
       'ALIAS_PRINCIPAL', 'POSTCODE', 'PRIVATE_STREET', 'LEGAL_PARCEL_ID',
       'CONFIDENCE', 'ADDRESS_SITE_PID', 'LEVEL_GEOCODED_CODE', 'PROPERTY_PID',
       'GNAF_PROPERTY_PID', 'PRIMARY_SECONDARY'],
      dtype='object')

In [ ]:
5035 in SA_ADDRESS_DETAIL_psv['POSTCODE'].unique()

In [ ]:
SA_ADDRESS_DETAIL_psv['POSTCODE'].unique().shape

In [43]:
locaility_points = SA_STREET_LOCALITY_POINT_psv[['STREET_LOCALITY_PID', 'LONGITUDE', 'LATITUDE']].merge(SA_ADDRESS_DETAIL_psv[['STREET_LOCALITY_PID', 'POSTCODE']].drop_duplicates(), on='STREET_LOCALITY_PID', how='left')
locaility_points.drop(columns=['STREET_LOCALITY_PID'], inplace=True)
locaility_points.dropna(inplace=True)
locaility_points.columns

Index(['LONGITUDE', 'LATITUDE', 'POSTCODE'], dtype='object')

In [ ]:
locaility_points.shape

In [44]:
postcode_coords = locaility_points[['LATITUDE', 'LONGITUDE']].to_numpy()
kdtree = KDTree(postcode_coords, metric='euclidean')

In [53]:
point_coords = df.to_numpy()

# Find index of nearest postcode point for each
distances, indices = kdtree.query(point_coords, k=1)  # k=1 → nearest
nearest_indices = indices.flatten()
nearest_distances = distances.flatten()

In [54]:
df['nearest_postcode'] = locaility_points.iloc[nearest_indices]['POSTCODE'].values

df['distance_km'] = nearest_distances*111  # Rough conversion factor for degrees to kilometers

In [55]:
df0 = df.query("distance_km <= 2.22").reset_index(drop=True)

In [56]:
df0['nearest_postcode'].unique().shape

(331,)

In [57]:
5035 in df0['nearest_postcode'].unique()

True

In [58]:
df0.to_csv('bom_postcodes_points.csv', index=False)